<a href="https://colab.research.google.com/github/furalmiqbadi/PembelajaranMesin_01_AbdulGhofurAlmiqbadi/blob/main/JS09/JS09_Tugas01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Langkah 1 - Install Library

In [2]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.7 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520035 sha256=a2d41b4506ee76bfa02fe837d090abcaf2d5431b32305cf5d3c71fc6dc80e527
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=f3bb5a1a1eb829c6c833fe34c44e9168d50833b72b631b44ee7e8a7ba3d4388e
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

## Langkah 2 - Import Library

In [3]:
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex
from sklearn.neighbors import NearestNeighbors

## Langkah 3 - Fungsi Bantu


In [4]:
def make_data(n, d, nq=100, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.random((n, d)).astype('float32')
    Q = rng.random((nq, d)).astype('float32')
    return X, Q


def normalize(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (X / norms).astype('float32')


def exact_search(X, Q, k, metric):
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric)
    nn.fit(X)
    start = time.time()
    _, idx = nn.kneighbors(Q)
    elapsed = time.time() - start
    return idx, elapsed


def run_annoy(X, Q, k, metric, n_trees=10):
    annoy_metric = 'euclidean' if metric == 'euclidean' else 'angular'
    index = AnnoyIndex(X.shape[1], annoy_metric)

    start = time.time()
    for i, v in enumerate(X):
        index.add_item(i, v)
    index.build(n_trees)
    build_time = time.time() - start

    start = time.time()
    ids = np.array([index.get_nns_by_vector(q, k) for q in Q])
    query_time = time.time() - start
    return ids, query_time, build_time


def run_faiss(X, Q, k, metric, nlist=100, nprobe=10):
    d = X.shape[1]
    nlist = min(nlist, max(1, len(X) // 10))

    if metric == 'euclidean':
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
        data, queries = X, Q
    else:
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
        data, queries = normalize(X), normalize(Q)

    start = time.time()
    index.train(data)
    index.add(data)
    build_time = time.time() - start

    index.nprobe = nprobe
    start = time.time()
    _, ids = index.search(queries, k)
    query_time = time.time() - start
    return ids, query_time, build_time


def run_hnsw(X, Q, k, metric, M=16, ef=50):
    d = X.shape[1]

    if metric == 'euclidean':
        space, data, queries = 'l2', X, Q
    else:
        space, data, queries = 'ip', normalize(X), normalize(Q)

    index = hnswlib.Index(space=space, dim=d)

    start = time.time()
    index.init_index(max_elements=len(data), ef_construction=200, M=M)
    index.add_items(data)
    build_time = time.time() - start

    index.set_ef(ef)
    start = time.time()
    ids, _ = index.knn_query(queries, k=k)
    query_time = time.time() - start
    return ids, query_time, build_time


def recall_at_k(pred, gt, k):
    pred = np.asarray(pred)
    gt = np.asarray(gt)
    return float(np.mean([
        len(set(pred[i][:k]).intersection(gt[i][:k])) / k
        for i in range(len(gt))
    ]))


## Langkah 4 - Jalankan Percobaan


In [5]:
N_LARGE = 1_000_000

scenarios = [
    (1_000, 2),
    (1_000, 5),
    (N_LARGE, 2),
    (N_LARGE, 5),
]
metrics = ['euclidean', 'cosine']
k = 10
nq = 100
rows = []

for n, d in scenarios:
    X, Q = make_data(n, d, nq=nq)

    for metric in metrics:
        gt_idx, t_exact = exact_search(X, Q, k, metric)
        rows.append({
            'Jumlah Data': n,
            'Dimensi': d,
            'Metrik': metric,
            'Algoritma': 'Exact NN',
            'Build (s)': 0.0,
            'Query (ms/query)': (t_exact * 1000) / nq,
            'Recall@10': 1.0,
        })

        runners = [
            ('ANNOY', run_annoy),
            ('FAISS', run_faiss),
            ('HNSW', run_hnsw),
        ]

        for name, runner in runners:
            idx, t_query, t_build = runner(X, Q, k, metric)
            rows.append({
                'Jumlah Data': n,
                'Dimensi': d,
                'Metrik': metric,
                'Algoritma': name,
                'Build (s)': t_build,
                'Query (ms/query)': (t_query * 1000) / nq,
                'Recall@10': recall_at_k(idx, gt_idx, k),
            })

        print(f"Selesai: n={n:,}, d={d}, metric={metric}")

hasil = pd.DataFrame(rows)
hasil['Build (s)'] = hasil['Build (s)'].round(3)
hasil['Query (ms/query)'] = hasil['Query (ms/query)'].round(4)
hasil['Recall@10'] = hasil['Recall@10'].round(4)
display(hasil)


Selesai: n=1,000, d=2, metric=euclidean
Selesai: n=1,000, d=2, metric=cosine
Selesai: n=1,000, d=5, metric=euclidean
Selesai: n=1,000, d=5, metric=cosine
Selesai: n=1,000,000, d=2, metric=euclidean
Selesai: n=1,000,000, d=2, metric=cosine
Selesai: n=1,000,000, d=5, metric=euclidean
Selesai: n=1,000,000, d=5, metric=cosine


,Jumlah Data,Dimensi,Metrik,Algoritma,Build (s),Query (ms/query),Recall@10
0,1000,2,euclidean,Exact NN,0.000,0.3655,1.000
1,1000,2,euclidean,ANNOY,0.031,0.0265,0.997
2,1000,2,euclidean,FAISS,0.007,0.0052,1.000
3,1000,2,euclidean,HNSW,0.054,0.0162,1.000
4,1000,2,cosine,Exact NN,0.000,0.0929,1.000
5,1000,2,cosine,ANNOY,0.030,0.0234,0.997
6,1000,2,cosine,FAISS,0.006,0.0045,1.000
7,1000,2,cosine,HNSW,0.046,0.0104,1.000
8,1000,5,euclidean,Exact NN,0.000,0.0161,1.000
9,1000,5,euclidean,ANNOY,0.014,0.0243,0.973


## Langkah 5 - Tabel Ringkasan



In [6]:
# Contoh pivot untuk metrik Euclidean.
pivot = hasil[hasil['Metrik'] == 'euclidean'].pivot_table(
    index=['Jumlah Data', 'Dimensi'],
    columns='Algoritma',
    values='Query (ms/query)',
)
display(pivot)


Algoritma             ANNOY  Exact NN   FAISS    HNSW
Jumlah Data Dimensi                                  
1000        2        0.0265    0.3655  0.0052  0.0162
            5        0.0243    0.0161  0.0094  0.0145
1000000     2        0.0598    4.9456  0.6083  0.0449
            5        0.0675    4.7774  0.8325  0.0464